### Llama index retriever

In [11]:
import os
import json
from typing import List, Optional
import asyncio
import warnings
import numpy as np

warnings.filterwarnings("ignore")

from llama_index.core import (
    VectorStoreIndex,
    SimpleDirectoryReader,
    Document,
    Settings,
    DocumentSummaryIndex,
    KeywordTableIndex
)

from llama_index.core.retrievers import (
    BaseRetriever,
    VectorIndexAutoRetriever,
    AutoMergingRetriever,
    RecursiveRetriever,
    QueryFusionRetriever,
)

from llama_index.core.node_parser import SentenceSplitter, HierarchicalNodeParser
from llama_index.core.schema import NodeWithScore, QueryBundle
from llama_index.core.query_engine import RetrieverQueryEngine
from llama_index.core.postprocessor import SentenceTransformerRerank
from llama_index.core.embeddings import BaseEmbedding
from llama_index.embeddings.huggingface import HuggingFaceEmbedding

from llama_index.core.indices.document_summary import (
    DocumentSummaryIndexLLMRetriever,
    DocumentSummaryIndexEmbeddingRetriever
)

from llama_index.retrievers.bm25 import BM25Retriever

from sentence_transformers import SentenceTransformer

from langchain_ollama import OllamaLLM

llm = OllamaLLM(
    model="qwen3:latest",
)

try:
    from scipy import stats
    SCIPY_AVAILABLE = True
except ImportError:
    SCIPY_AVAILABLE = False
    print("⚠️ scipy not available - some advanced fusion features will be limited")

print("✅ All imports successful!")

✅ All imports successful!


In [13]:
embed_model = HuggingFaceEmbedding(
    model_name = "ibm-granite/granite-embedding-97m-multilingual-r2"
)

## Global settings

Settings.llm = llm
Settings.embed_model = embed_model


Loading weights: 100%|██████████| 74/74 [00:00<00:00, 14792.61it/s]


In [15]:
SAMPLE_DOCUMENTS = [
    "Machine learning is a subset of artificial intelligence that focuses on algorithms that can learn from data.",
    "Deep learning uses neural networks with multiple layers to model and understand complex patterns in data.",
    "Natural language processing enables computers to understand, interpret, and generate human language.",
    "Computer vision allows machines to interpret and understand visual information from the world.",
    "Reinforcement learning is a type of machine learning where agents learn to make decisions through rewards and penalties.",
    "Supervised learning uses labeled training data to learn a mapping from inputs to outputs.",
    "Unsupervised learning finds hidden patterns in data without labeled examples.",
    "Transfer learning leverages knowledge from pre-trained models to improve performance on new tasks.",
    "Generative AI can create new content including text, images, code, and more.",
    "Large language models are trained on vast amounts of text data to understand and generate human-like text."
]

# Consistent query examples used throughout the lab
DEMO_QUERIES = {
    "basic": "What is machine learning?",
    "technical": "neural networks deep learning", 
    "learning_types": "different types of learning",
    "advanced": "How do neural networks work in deep learning?",
    "applications": "What are the applications of AI?",
    "comprehensive": "What are the main approaches to machine learning?",
    "specific": "supervised learning techniques"
}

### Initialize the lab

In [18]:
class AdvancedRetriverLab:
    def __init__(self):
        self.documents = [Document(text = text) for text in SAMPLE_DOCUMENTS]
        self.nodes = SentenceSplitter().get_nodes_from_documents(self.documents)

        self.vector_index = VectorStoreIndex.from_documents(self.documents)
        self.document_summary_index = DocumentSummaryIndex.from_documents(self.documents)
        self.keyword_index = KeywordTableIndex.from_documents(self.documents)

        print("✅ Advanced Retrievers Lab Initialized!")
        print(f"📄 Loaded {len(self.documents)} documents")
        print(f"🔢 Created {len(self.nodes)} nodes")

# Initialize the lab
lab = AdvancedRetriverLab()

current doc id: 11062c6f-068e-4d0c-8db8-94103d3abd32
current doc id: c13931c9-9f9c-4363-9d26-5794ea4ea6e6
current doc id: 5f4758a0-3d29-4341-a5e0-c0b9268dd56d
current doc id: 71966b85-3996-402c-a04c-567556d46f62
current doc id: 74a16edb-3c81-4232-acda-62f8aada942e
current doc id: e7ddb38b-7cc4-454f-a12d-210934983336
current doc id: a49ab05f-6a96-4a15-b2cb-2ad37813a3f2
current doc id: deb4f437-d65d-4cc6-b167-e1a8914c35d8
current doc id: 163b5d4d-23fd-4f10-8805-0ced23a097b3
current doc id: 1af91d9e-5bca-478a-bf70-762c4c035760
✅ Advanced Retrievers Lab Initialized!
📄 Loaded 10 documents
🔢 Created 10 nodes


### Vector Index Retriever

In [21]:
from llama_index.core.retrievers import (
    VectorIndexRetriever,
)

vector_retriver = VectorIndexRetriever(
    index= lab.vector_index,
    similarity_top_k= 3
)

# Other way to create the same
alt = lab.vector_index.as_retriever(similarity_top_k = 3)

query = DEMO_QUERIES["basic"]

nodes = vector_retriver.retrieve(query)

alt_nodes = alt.retrieve(query)

for i, node in enumerate(nodes, 1):
    print(f"{i}. Score: {node.score:.4f}")
    print(f"   Text: {node.text[:100]}...")
    print()

for i, node in enumerate(alt_nodes, 1):
    print(f"{i}. Score: {node.score:.4f}")
    print(f"   Text: {node.text[:100]}...")
    print()

1. Score: 0.9720
   Text: Machine learning is a subset of artificial intelligence that focuses on algorithms that can learn fr...

2. Score: 0.8857
   Text: Reinforcement learning is a type of machine learning where agents learn to make decisions through re...

3. Score: 0.8765
   Text: Deep learning uses neural networks with multiple layers to model and understand complex patterns in ...

1. Score: 0.9720
   Text: Machine learning is a subset of artificial intelligence that focuses on algorithms that can learn fr...

2. Score: 0.8857
   Text: Reinforcement learning is a type of machine learning where agents learn to make decisions through re...

3. Score: 0.8765
   Text: Deep learning uses neural networks with multiple layers to model and understand complex patterns in ...



### BM25 Retriever - Advanced Keyword-Based Search

In [22]:
import Stemmer

bm25_retriver = BM25Retriever.from_defaults(
    nodes = lab.nodes,
    similarity_top_k=3,
    stemmer=Stemmer.Stemmer("english"),
    language="english"
)

query = DEMO_QUERIES["technical"]
nodes = bm25_retriver.retrieve(query)

for i, node in enumerate(nodes, 1):
    score = node.score if hasattr(node, 'score') and node.score else 0
    print(f"{i}. BM25 Score: {score:.4f}")
    print(f"   Text: {node.text[:100]}...")
    
    # Highlight which query terms appear in the text
    text_lower = node.text.lower()
    query_terms = query.lower().split()
    found_terms = [term for term in query_terms if term in text_lower]
    if found_terms:
        print(f"   → Found terms: {found_terms}")
    print()

1. BM25 Score: 2.5203
   Text: Deep learning uses neural networks with multiple layers to model and understand complex patterns in ...
   → Found terms: ['neural', 'networks', 'deep', 'learning']

2. BM25 Score: 0.3372
   Text: Reinforcement learning is a type of machine learning where agents learn to make decisions through re...
   → Found terms: ['learning']

3. BM25 Score: 0.3024
   Text: Machine learning is a subset of artificial intelligence that focuses on algorithms that can learn fr...
   → Found terms: ['learning']



### Document Summary Index Retrievers

In [24]:
doc_summary_retriver_llm = DocumentSummaryIndexLLMRetriever(
    lab.document_summary_index,
    choice_top_k=3
)

doc_summary_retriver_embedding = DocumentSummaryIndexEmbeddingRetriever(
    lab.document_summary_index,
    similarity_top_k= 3
)

query = DEMO_QUERIES["learning_types"]

nodes_llm = doc_summary_retriver_llm.retrieve(query)
nodes_embed = doc_summary_retriver_embedding.retrieve(query)

for i, node in enumerate(nodes_llm[:2], 1):
        print(f"{i}. Score: {node.score:.4f}" if hasattr(node, 'score') and node.score else f"{i}. (Document summary)")
        print(f"   Text: {node.text[:80]}...")
        print()

for i, node in enumerate(nodes_embed[:2], 1):
        print(f"{i}. Score: {node.score:.4f}" if hasattr(node, 'score') and node.score else f"{i}. (Document summary)")
        print(f"   Text: {node.text[:80]}...")
        print()

1. Score: 9.0000
   Text: Supervised learning uses labeled training data to learn a mapping from inputs to...

2. Score: 8.0000
   Text: Unsupervised learning finds hidden patterns in data without labeled examples....

1. (Document summary)
   Text: Reinforcement learning is a type of machine learning where agents learn to make ...

2. (Document summary)
   Text: Machine learning is a subset of artificial intelligence that focuses on algorith...



### Auto Merging Retriever - Hierarchical Context Preservation

Uses hierarchical chunking to break documents into parent and child nodes
Retrieves parent if enough children match - intelligent merging logic
Preserves context in long documents by consolidating related content
Dual Storage: Smaller child chunks are indexed in the vector store for precise matching, while larger parent chunks are stored in the docstore

In [26]:
node_parser = HierarchicalNodeParser.from_defaults(
    chunk_sizes=[512, 256, 128]
)

hier_node = node_parser.get_nodes_from_documents(
    lab.documents
)

from llama_index.core import StorageContext
from llama_index.core.storage.docstore import SimpleDocumentStore
from llama_index.core.vector_stores import SimpleVectorStore

docstore = SimpleDocumentStore() # to store parent
docstore.add_documents(hier_node)

storage_context = StorageContext.from_defaults(docstore=docstore)

base_index = VectorStoreIndex(hier_node, storage_context=storage_context)
base_retriver = base_index.as_retriever(similarity_tok_k = 6)

auto_merging_retriver = AutoMergingRetriever(
    base_retriver,
    storage_context,
    verbose=True
)

query = DEMO_QUERIES["advanced"]  # "How do neural networks work in deep learning?"
nodes = auto_merging_retriver.retrieve(query)

for i, node in enumerate(nodes[:3], 1):
    print(f"{i}. Score: {node.score:.4f}" if hasattr(node, 'score') and node.score else f"{i}. (Auto-merged)")
    print(f"   Text: {node.text[:120]}...")
    print()

> Merging 1 nodes into parent node.
> Parent node id: ee046546-152b-4ea6-8bf0-a62b2319dc32.
> Parent node text: Deep learning uses neural networks with multiple layers to model and understand complex patterns ...

1. Score: 0.9192
   Text: Deep learning uses neural networks with multiple layers to model and understand complex patterns in data....



### Recursive Retriever - Multi-Level Reference Following

How it works (from authoritative source):

    Follows node references - traverses relationships to find referenced content
    Supports chunk and metadata linking - handles different types of references
    Multi-Level Navigation: Can execute sub-queries on referenced retrievers or query engines
    Network Building: Creates a network of interconnected retrievers that can reference each other


In [28]:
# We need to create document with reference
docs_with_refs = []

for i, doc in enumerate(lab.documents):
    ref_doc = Document(
        text = doc.text,
        metadata = {
            "doc_id": f"doc_{i}",
            "references" : [f"doc_{j}" for j in range(len(lab.documents)) if j != i][:2]
        }
    )
    docs_with_refs.append(ref_doc)

# create index with ref docs
ref_index = VectorStoreIndex.from_documents(docs_with_refs)

retrive_dict = {
    f"doc_{i}": ref_index.as_retriever(similarity_top_k=1) for i in range(len(docs_with_refs))
}

base_retriver = ref_index.as_retriever(similarity_top_k = 2)

retrive_dict["vector"] = base_retriver

recursive_retriver = RecursiveRetriever(
    "vector",
    retriever_dict=retrive_dict,
    query_engine_dict={},
    verbose=True
)

query = DEMO_QUERIES["applications"]

nodes = recursive_retriver.retrieve(query)

for i, node in enumerate(nodes[:3], 1):
    print(f"{i}. Score: {node.score:.4f}" if hasattr(node, 'score') and node.score else f"{i}. (Recursive)")
    print(f"   Text: {node.text[:100]}...")
    print()


Retrieving with query id None: What are the applications of AI?
Retrieving text node: Generative AI can create new content including text, images, code, and more.
Retrieving text node: Machine learning is a subset of artificial intelligence that focuses on algorithms that can learn from data.
1. Score: 0.8211
   Text: Generative AI can create new content including text, images, code, and more....

2. Score: 0.8158
   Text: Machine learning is a subset of artificial intelligence that focuses on algorithms that can learn fr...



### Query Fusion Retriever - Multi-Query Enhancement with Advanced Fusion

How it works (from authoritative source):

    Combines results from multiple retrievers - e.g., vector-based and keyword-based methods
    Supports multiple query variations - generates different formulations of the same query
    Uses fusion strategies to improve recall - sophisticated merging techniques
    Improved Coverage: Reduces impact of query formulation on final results


In [29]:
base_retriver = lab.vector_index.as_retriever(similarity_top_k = 3)

query = DEMO_QUERIES["comprehensive"]


### 6.1 Reciprocal Rank Fusion (RRF) Mode


How it works within QueryFusionRetriever:

    Generates multiple query variations (e.g., "machine learning approaches", "ML techniques", "learning algorithms")
    Retrieves results for each query variation
    Calculates reciprocal rank score: 1 / (rank + k) where k is typically 60
    Sums reciprocal rank scores across all query variations for each document
    Re-ranks documents by combined RRF scores

Mathematical formula:

RRF_score(d) = Σ (1 / (rank_i(d) + k))


In [30]:
rrf_query_fusion = QueryFusionRetriever(
    [base_retriver],
    similarity_top_k=3,
    num_queries=3,
    mode="reciprocal_rerank",
    use_async=False,
    verbose=True
)

nodes = rrf_query_fusion.retrieve(query)

for i, node in enumerate(nodes, 1):
    print(f"{i}. Final RRF Score: {node.score:.4f}")
    print(f"   Text: {node.text[:100]}...")
    print()

Generated queries:
What are the primary categories of machine learning approaches?
What are the different paradigms in machine learning?
1. Final RRF Score: 0.0500
   Text: Machine learning is a subset of artificial intelligence that focuses on algorithms that can learn fr...

2. Final RRF Score: 0.0492
   Text: Deep learning uses neural networks with multiple layers to model and understand complex patterns in ...

3. Final RRF Score: 0.0323
   Text: Supervised learning uses labeled training data to learn a mapping from inputs to outputs....



### 6.2 Relative Score Fusion Mode

How it works within QueryFusionRetriever:

    Generates multiple query variations using LLM
    Retrieves results for each query variation
    Normalizes each query's scores by dividing by the maximum score in that query's results
    Creates scores in the range [0, 1] where 1 is the best result from each query variation
    Combines normalized scores using weighted average or sum

Mathematical approach:

normalized_score_i(d) = score_i(d) / max_score_i
combined_score(d) = Σ (weight_i × normalized_score_i(d))


In [31]:
base_retriever = lab.vector_index.as_retriever(similarity_top_k=5)

query = DEMO_QUERIES["comprehensive"] 

rel_score_fusion = QueryFusionRetriever(
        [base_retriever],
        similarity_top_k=3,
        num_queries=3,
        mode="relative_score",
        use_async=False,
        verbose=False
    )

nodes = rel_score_fusion.retrieve(query)

In [32]:
for i, node in enumerate(nodes, 1):
    print(f"{i}. Combined Relative Score: {node.score:.4f}")
    print(f"   Text: {node.text[:100]}...")
    print()

1. Combined Relative Score: 1.0000
   Text: Machine learning is a subset of artificial intelligence that focuses on algorithms that can learn fr...

2. Combined Relative Score: 0.3638
   Text: Deep learning uses neural networks with multiple layers to model and understand complex patterns in ...

3. Combined Relative Score: 0.2632
   Text: Supervised learning uses labeled training data to learn a mapping from inputs to outputs....



### 6.3 Distribution-Based Score Fusion Mode

How it works within QueryFusionRetriever:

    Generates multiple query variations using LLM
    Analyzes the statistical distribution of scores from each query variation
    Normalizes scores using distribution parameters (mean, standard deviation, percentiles)
    Applies statistical transformations like z-score normalization or percentile ranking
    Combines normalized scores with confidence weighting based on distribution characteristics

Statistical approaches used:

    Z-score normalization: Centers scores around mean with unit variance
        Formula: z_score = (score - mean) / std_dev
        Converts to [0,1] range using sigmoid: 1 / (1 + exp(-z_score))

    Percentile ranking: Converts scores to percentile positions
        Formula: percentile = rank(score) / total_results

    Distribution-aware normalization: Considers score distribution shape
        Uses IQR (Interquartile Range) to adjust for distribution spread
        Handles multi-modal distributions from different query variations


In [33]:
base_retriever = lab.vector_index.as_retriever(similarity_top_k=8)

# Use the same query for consistency across all fusion modes
query = DEMO_QUERIES["comprehensive"]  # "What are the main approaches to machine learning?"

# Create query fusion retriever with distribution-based mode
dist_fusion = QueryFusionRetriever(
    [base_retriever],
    similarity_top_k=3,
    num_queries=3,
    mode="dist_based_score",
    use_async=False,
    verbose=False
)

nodes = dist_fusion.retrieve(query)
    
for i, node in enumerate(nodes, 1):
    print(f"{i}. Statistically Normalized Score: {node.score:.4f}")
    print(f"   Text: {node.text[:100]}...")
    print()

1. Statistically Normalized Score: 0.8535
   Text: Machine learning is a subset of artificial intelligence that focuses on algorithms that can learn fr...

2. Statistically Normalized Score: 0.6032
   Text: Deep learning uses neural networks with multiple layers to model and understand complex patterns in ...

3. Statistically Normalized Score: 0.5438
   Text: Reinforcement learning is a type of machine learning where agents learn to make decisions through re...

